# Mini Transformer From Scratch Workshop

> 从 0 手写一个可训练、可生成的最小 Transformer 语言模型（PyTorch）

这个 Notebook 覆盖：

1. 构建最小字符级数据集与 tokenizer。
2. 手写 Causal Self-Attention（含 mask）。
3. 组装 Transformer Block 与 LM Head。
4. 完整训练循环（loss / perplexity）。
5. 采样生成与注意力可视化。

建议：每段代码先自己写一版，再对照这里的实现。

In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", DEVICE)

## Part A. 数据与 Tokenizer

这里用一个小语料做字符级建模，重点理解：

- 语言建模任务：给前文，预测下一个 token。
- `block_size` 的含义：模型一次可见的最大上下文长度。

In [ ]:
text = (
    "deep learning changes software. "
    "transformers learn context with attention. "
    "causal masking prevents information leak. "
    "we train models by minimizing cross entropy loss. "
    "good engineering means reproducibility and debugging. "
) * 100

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join([itos[i] for i in ids])

data = torch.tensor(encode(text), dtype=torch.long)

print("vocab_size:", vocab_size)
print("data shape:", data.shape)
print("sample vocab:", chars[:20])
print("sample decode:", decode(data[:80].tolist()))

In [ ]:
@dataclass
class Config:
    block_size: int = 64
    batch_size: int = 64
    n_embed: int = 128
    n_head: int = 4
    n_layer: int = 4
    dropout: float = 0.1
    lr: float = 3e-4
    weight_decay: float = 1e-2
    max_steps: int = 800
    eval_interval: int = 80

cfg = Config()


class CharLMDataset(Dataset):
    def __init__(self, token_tensor: torch.Tensor, block_size: int):
        self.data = token_tensor
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size - 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y


split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]

train_ds = CharLMDataset(train_data, cfg.block_size)
val_ds = CharLMDataset(val_data, cfg.block_size)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=False)

xb, yb = next(iter(train_loader))
print("x batch:", xb.shape, "y batch:", yb.shape)
print("x[0]:", decode(xb[0].tolist()[:40]))
print("y[0]:", decode(yb[0].tolist()[:40]))

## Part B. 手写 Transformer 模块

我们实现一个 Pre-LN 风格的 Decoder-only Transformer：

- Token Embedding + Position Embedding
- Causal Self-Attention（下三角 mask）
- FFN
- 残差连接 + LayerNorm
- LM Head 输出 logits

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embed: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        assert n_embed % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embed // n_head
        self.block_size = block_size

        self.qkv_proj = nn.Linear(n_embed, 3 * n_embed)
        self.out_proj = nn.Linear(n_embed, n_embed)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        B, T, C = x.shape

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(C, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, H, T, Dh)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, H, T, T)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        y = att @ v  # (B, H, T, Dh)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.out_proj(y))

        if return_attn:
            return y, att
        return y


class FeedForward(nn.Module):
    def __init__(self, n_embed: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.GELU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, n_embed: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = CausalSelfAttention(n_embed, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embed)
        self.ffn = FeedForward(n_embed, dropout)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        if return_attn:
            attn_out, attn = self.attn(self.ln1(x), return_attn=True)
            x = x + attn_out
            x = x + self.ffn(self.ln2(x))
            return x, attn
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.block_size = cfg.block_size

        self.tok_emb = nn.Embedding(vocab_size, cfg.n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embed)
        self.drop = nn.Dropout(cfg.dropout)

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(cfg.n_embed, cfg.n_head, cfg.block_size, cfg.dropout)
                for _ in range(cfg.n_layer)
            ]
        )

        self.ln_f = nn.LayerNorm(cfg.n_embed)
        self.lm_head = nn.Linear(cfg.n_embed, vocab_size, bias=False)

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
        return_attn_layer: Optional[int] = None,
    ):
        B, T = idx.shape
        if T > self.block_size:
            raise ValueError(f"T={T} exceeds block_size={self.block_size}")

        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))[None, :, :]
        x = self.drop(tok + pos)

        attn_to_return = None
        for i, blk in enumerate(self.blocks):
            if return_attn_layer is not None and i == return_attn_layer:
                x, attn_to_return = blk(x, return_attn=True)
            else:
                x = blk(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        if return_attn_layer is not None:
            return logits, loss, attn_to_return
        return logits, loss

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int = 100, temperature: float = 1.0, top_k: Optional[int] = None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

In [ ]:
model = MiniTransformerLM(vocab_size=vocab_size, cfg=cfg).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(f"params: {num_params/1e6:.3f}M")

x_demo, y_demo = next(iter(train_loader))
x_demo, y_demo = x_demo.to(DEVICE), y_demo.to(DEVICE)
logits_demo, loss_demo = model(x_demo, y_demo)
print("logits shape:", logits_demo.shape)
print("demo loss:", float(loss_demo))

## Part C. 训练循环

我们采用标准 next-token 训练：

- 输入 `x[:, 0:T]`
- 目标 `y[:, 1:T+1]`
- loss 使用 token 级交叉熵
- 指标补充 perplexity = `exp(loss)`

In [ ]:
def cycle(loader):
    while True:
        for batch in loader:
            yield batch


def estimate_loss(model, loader, eval_batches: int = 30):
    model.eval()
    losses = []
    with torch.no_grad():
        for i, (xb, yb) in enumerate(loader):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            _, loss = model(xb, yb)
            losses.append(loss.item())
            if i + 1 >= eval_batches:
                break
    model.train()
    return float(np.mean(losses))


optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
train_iter = cycle(train_loader)

steps, train_loss_hist, val_loss_hist, ppl_hist = [], [], [], []

for step in range(1, cfg.max_steps + 1):
    xb, yb = next(train_iter)
    xb, yb = xb.to(DEVICE), yb.to(DEVICE)

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % cfg.eval_interval == 0 or step == 1:
        train_loss = estimate_loss(model, train_loader, eval_batches=20)
        val_loss = estimate_loss(model, val_loader, eval_batches=20)
        ppl = math.exp(val_loss)

        steps.append(step)
        train_loss_hist.append(train_loss)
        val_loss_hist.append(val_loss)
        ppl_hist.append(ppl)

        print(f"step={step:04d} train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_ppl={ppl:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(steps, train_loss_hist, label="train")
axes[0].plot(steps, val_loss_hist, label="val")
axes[0].set_title("Loss Curve")
axes[0].set_xlabel("step")
axes[0].set_ylabel("cross entropy")
axes[0].legend()

axes[1].plot(steps, ppl_hist)
axes[1].set_title("Validation Perplexity")
axes[1].set_xlabel("step")
axes[1].set_ylabel("ppl")

plt.tight_layout()
plt.show()

## Part D. 采样生成

我们从一个 prompt 出发，比较不同 `temperature / top_k` 的生成风格。

In [ ]:
def sample_text(prompt: str, temperature: float = 1.0, top_k: Optional[int] = None, max_new_tokens: int = 180):
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    out_ids = model.generate(ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(out_ids[0].tolist())

prompt = "transformers "

print("=== temperature=0.8, top_k=8 ===")
print(sample_text(prompt, temperature=0.8, top_k=8, max_new_tokens=160))
print()
print("=== temperature=1.1, top_k=20 ===")
print(sample_text(prompt, temperature=1.1, top_k=20, max_new_tokens=160))

## Part E. 注意力可视化

观察某一层某个 head 的注意力矩阵：

- 横轴：被关注的位置 (key)
- 纵轴：当前查询位置 (query)
- 上三角应被 mask（接近 0）

In [ ]:
inspect_text = "transformers learn context with attention. "
inspect_ids = torch.tensor([encode(inspect_text[: cfg.block_size])], dtype=torch.long, device=DEVICE)

model.eval()
with torch.no_grad():
    logits, loss, attn = model(inspect_ids, inspect_ids, return_attn_layer=0)

# attn shape: (B, H, T, T)
attn_map = attn[0, 0].detach().cpu().numpy()  # 第0个样本，第0个head
T = attn_map.shape[0]

plt.figure(figsize=(5.5, 4.5))
plt.imshow(attn_map, cmap="viridis", aspect="auto")
plt.colorbar()
plt.title("Layer 0 Head 0 Attention Map")
plt.xlabel("Key Position")
plt.ylabel("Query Position")
plt.show()

# 验证上三角 mask：取上三角平均值（理想接近 0）
upper = np.triu(attn_map, k=1)
print("mean upper-triangle attention:", float(upper.mean()))

## Part F. 进阶练习

1. **模型容量实验**：将 `n_embed` 改为 `64 / 192`，对比 loss 与生成质量。
2. **上下文长度实验**：将 `block_size` 改为 `32 / 128`，观察长依赖效果。
3. **正则化实验**：将 `dropout` 设为 `0.0 / 0.2`，观察过拟合。
4. **优化器实验**：用 `SGD+Momentum` 替代 `AdamW`，比较收敛速度。
5. **采样策略实验**：固定 prompt，对比 `greedy / top-k / temperature` 的文本多样性。

如果你把这些练习做完，Transformer 的训练与推理机制会非常扎实。